# Demo — Test Use Cases (Notebook 6)

Mục tiêu notebook này là mô phỏng flow dùng thử theo từng bước, hiển thị rõ **Input** và **Output**. Runtime web hiện tại dùng **artifact recommender** làm luồng search chính; các biến thể multi-field trong notebook này chỉ còn giá trị minh hoạ/thảo luận.

- **Flow 1 (artifact-aligned)**: *User search text* → *Retrieval* (TF‑IDF trên `review_profile`) → *ABSA* (rerank top‑N)
- **Flow 2 (thực nghiệm cũ để tham khảo)**: *Multi-field search* (title/genre/semantic) → *Weighted re-rank* → *(optional) ABSA bonus* (movie-level profiles)

Notebook có 4 use-cases minh hoạ:
- **A**: Query tên phim (có thể gõ sai nhẹ)
- **B**: Query theo thể loại
- **C**: Query theo ngữ cảnh dài (semantic)
- **D**: Query có cảm xúc/aspect để thấy ABSA bonus thay đổi thứ hạng

Lưu ý: toàn bộ demo dùng artifacts & file trong `Notebook_Report/`. Checkpoint ABSA mặc định: `absa/artifacts/absa_distilroberta_latest/`; trong thực tế repo này đang dùng artifact export từ `Kaggle_ABSA_Train_Standalone.ipynb`, đổi bằng biến môi trường `ABSA_ARTIFACT_NAME` nếu cần.


In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# -----------------------------
# Cấu hình đường dẫn
# -----------------------------
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "Notebook_Report":
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "Notebook_Report" / "cleaned_profiles.csv").exists():
            NOTEBOOK_DIR = p / "Notebook_Report"
            break

DATA_CLEANED_PROFILES = NOTEBOOK_DIR / "cleaned_profiles.csv"
ABSA_ARTIFACT_DIR = NOTEBOOK_DIR / "absa" / "artifacts" / os.environ.get(
    "ABSA_ARTIFACT_NAME", "absa_distilroberta_latest"
)
ABSA_CKPT_PATH = ABSA_ARTIFACT_DIR / "model.pt"

# -----------------------------
# Input demo: user search
# -----------------------------
USER_QUERY = "great visuals but pacing was too slow"

# Retrieval
TOP_K_RETRIEVAL = 30
TOP_K_FINAL = 10

# ABSA rerank: chỉ chạy ABSA trên top-N sau retrieval để tiết kiệm thời gian
TOP_N_FOR_ABSA = 20

# ABSA target label: lấy nhãn có probability cao nhất của query
TARGET_LABEL_TOP1 = True

print("Notebook dir:", NOTEBOOK_DIR)
print("Loaded profiles:", DATA_CLEANED_PROFILES.exists(), DATA_CLEANED_PROFILES)
print("ABSA ckpt exists:", ABSA_CKPT_PATH.exists(), ABSA_CKPT_PATH)

print("\n=== User Input ===")
print(USER_QUERY)


Notebook dir: /Users/kotori/CineSen/Notebook_Report
Loaded profiles: True /Users/kotori/CineSen/Notebook_Report/cleaned_profiles.csv
ABSA ckpt exists: True /Users/kotori/CineSen/Notebook_Report/absa/artifacts/absa_distilroberta_latest/model.pt

=== User Input ===
great visuals but pacing was too slow


In [2]:
# -----------------------------
# Bước 1: Load dữ liệu đã clean
# Output: danh sách phim + review_profile để làm corpus retrieval + ABSA
# -----------------------------
if not DATA_CLEANED_PROFILES.exists():
    raise FileNotFoundError(
        f"Không thấy {DATA_CLEANED_PROFILES}. Hãy chạy `02_Data_Preprocessing_EDA.ipynb` để tạo cleaned_profiles.csv trước."
    )

cols_needed = ["tmdb_id", "title", "review_profile"]
df = pd.read_csv(DATA_CLEANED_PROFILES)
missing = [c for c in cols_needed if c not in df.columns]
if missing:
    raise ValueError(f"cleaned_profiles.csv thiếu cột: {missing}")

df["review_profile"] = df["review_profile"].fillna("").astype(str).str.strip()
df = df[df["review_profile"].str.len() > 0].reset_index(drop=True)

print("Loaded movies:", len(df))
print("Sample rows:")
display(df.head(3)[["tmdb_id", "title", "review_profile"]])


Loaded movies: 3230
Sample rows:


,tmdb_id,title,review_profile
0,875828,Peaky Blinders: The Immortal Man,anyone remember michael elphick s private schu...
1,83533,Avatar: Fire and Ash,full spoiler free review avatar fire and ash l...
2,687163,Project Hail Mary,when times are tough and world weary souls hav...


In [3]:
# -----------------------------
# Bước 2: Retrieval (TF-IDF) dựa trên review_profile
# Input: USER_QUERY
# Output: TOP_K_RETRIEVAL phim ứng viên + score
# -----------------------------
from sklearn.feature_extraction.text import TfidfVectorizer

# Theo eval_results.json, model tốt nhất: tfidf_uni_bigram_min2
# ngram_range=(1,2), min_df=2
VECTORIZER = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=1.0)

corpus = df["review_profile"].tolist()
X = VECTORIZER.fit_transform(corpus)

q_vec = VECTORIZER.transform([USER_QUERY])

# cosine similarity cho TF-IDF: dùng dot vì TF-IDF đã chuẩn hoá
scores = (X @ q_vec.T).toarray().squeeze(1)

a = np.argsort(-scores)[:TOP_K_RETRIEVAL]

candidates = df.iloc[a].copy()
candidates["retrieval_score"] = scores[a]

print("\n=== Retrieval Output ===")
print("Top candidates:")
display(candidates[["tmdb_id", "title", "retrieval_score"]].head(10))



=== Retrieval Output ===
Top candidates:


,tmdb_id,title,retrieval_score
712,11827,Heavy Metal,0.059670
2040,638,Lost Highway,0.059107
2212,166424,Fantastic Four,0.048368
2360,1254786,The Life List,0.048192
385,62,2001: A Space Odyssey,0.046938
3174,664469,Amsterdam,0.044191
2395,301351,We Are Your Friends,0.043905
2850,441282,Night Hunter,0.039813
188,315837,Ghost in the Shell,0.039557
989,400155,Hotel Transylvania 3: Summer Vacation,0.039146


In [4]:
# -----------------------------
# Bước 3: Load ABSA checkpoint + define inference
# Input: ABSA artifacts từ Notebook 04
# Output: hàm infer_absa(text) -> probs[aspect×sentiment]
# -----------------------------
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

ASPECTS = ["script", "acting", "visuals", "music", "pacing", "direction", "overall"]
SENTIMENTS = ["negative", "neutral", "positive"]
NUM_LABELS = len(ASPECTS) * len(SENTIMENTS)


def get_label_index(aspect: str, sentiment: str) -> int:
    if aspect not in ASPECTS or sentiment not in SENTIMENTS:
        raise ValueError(f"Unknown aspect={aspect!r} or sentiment={sentiment!r}")
    return ASPECTS.index(aspect) * len(SENTIMENTS) + SENTIMENTS.index(sentiment)


class AbsaClassifier(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.head = nn.Linear(self.backbone.config.hidden_size, NUM_LABELS)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.head(cls)


def load_absa_model():
    if not ABSA_CKPT_PATH.exists():
        raise FileNotFoundError(
            f"Không thấy checkpoint ABSA: {ABSA_CKPT_PATH}. Hãy chạy `04_Advanced_ABSA_Modeling.ipynb` để export checkpoint trước."
        )

    ckpt = torch.load(ABSA_CKPT_PATH, map_location="cpu")
    model_name = ckpt.get("model_name", "distilroberta-base")
    max_len = int(ckpt.get("max_length", 256))

    model = AbsaClassifier(model_name)
    model.load_state_dict(ckpt["state_dict"], strict=True)

    tok_dir = ABSA_ARTIFACT_DIR / "tokenizer"
    if tok_dir.exists():
        tokenizer = AutoTokenizer.from_pretrained(tok_dir)
    else:
        tokenizer = AutoTokenizer.from_pretrained(model_name)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    thr = float(ckpt.get("threshold", 0.5))
    return model, tokenizer, device, max_len, thr


absa_model, absa_tokenizer, absa_device, MAX_LENGTH, THRESHOLD = load_absa_model()


def infer_absa_probs(text: str):
    enc = absa_tokenizer(
        text,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    input_ids = enc["input_ids"].to(absa_device)
    attention_mask = enc["attention_mask"].to(absa_device)

    with torch.no_grad():
        logits = absa_model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy().squeeze(0)

    return probs


print("Loaded ABSA model on:", absa_device)
print("Checkpoint:", ABSA_CKPT_PATH)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded ABSA model on: cpu
Checkpoint: /Users/kotori/CineSen/Notebook_Report/absa/artifacts/absa_distilroberta_latest/model.pt


In [5]:
# -----------------------------
# Bước 4: ABSA filter cho query
# Input: USER_QUERY
# Output: target aspect×sentiment và top labels
# -----------------------------
query_probs = infer_absa_probs(USER_QUERY)

# top-5 labels theo probability
label_names = [(a, s) for a in ASPECTS for s in SENTIMENTS]
ranked = []
for idx, p in enumerate(query_probs):
    a, s = label_names[idx]
    ranked.append((p, a, s, idx))
ranked.sort(reverse=True, key=lambda x: x[0])

top5 = ranked[:5]
print("\n=== ABSA on Query (top labels) ===")
for p, a, s, idx in top5:
    print(f"{a}:{s} -> {p:.3f}")

if TARGET_LABEL_TOP1:
    target_p, target_aspect, target_sentiment, target_idx = top5[0]
else:
    # fallback: chọn overall theo argmax
    target_aspect, target_sentiment = "overall", SENTIMENTS[int(np.argmax(query_probs[[get_label_index('overall', s) for s in SENTIMENTS]]))]
    target_idx = get_label_index(target_aspect, target_sentiment)
    target_p = float(query_probs[target_idx])

print("\nTarget label cho rerank:", f"{target_aspect}:{target_sentiment}", "prob=", f"{target_p:.3f}")



=== ABSA on Query (top labels) ===
pacing:positive -> 0.924
overall:positive -> 0.897
visuals:positive -> 0.741
script:positive -> 0.651
pacing:neutral -> 0.282

Target label cho rerank: pacing:positive prob= 0.924


In [6]:
# -----------------------------
# Bước 5: ABSA rerank trên top-N retrieval
# Input: candidates review_profile
# Output: top K cuối cùng theo ABSA score
# -----------------------------

def absa_score_for_movie(text: str, target_idx: int):
    probs = infer_absa_probs(text)
    return float(probs[target_idx])

# Lấy profile cho ABSA
candidate_texts = candidates.head(TOP_N_FOR_ABSA).copy()

absa_scores = []
for i, row in candidate_texts.iterrows():
    txt = row["review_profile"]
    score = absa_score_for_movie(txt, target_idx)
    absa_scores.append(score)

candidate_texts["absa_target_score"] = absa_scores
candidate_texts = candidate_texts.sort_values(by="absa_target_score", ascending=False)

final = candidate_texts.head(TOP_K_FINAL).copy()

print("\n=== Final Output (Retrieval + ABSA rerank) ===")
print("Rerank target:", f"{target_aspect}:{target_sentiment}")

display(final[["tmdb_id", "title", "retrieval_score", "absa_target_score"]])



=== Final Output (Retrieval + ABSA rerank) ===
Rerank target: pacing:positive


,tmdb_id,title,retrieval_score,absa_target_score
2913,68722,The Master,0.039145,0.921712
2842,559907,The Green Knight,0.038205,0.883070
935,4982,American Gangster,0.036527,0.872717
2944,491480,The Boy Who Harnessed the Wind,0.037100,0.869706
3174,664469,Amsterdam,0.044191,0.774908
385,62,2001: A Space Odyssey,0.046938,0.752695
3057,635731,Pig,0.035232,0.686212
188,315837,Ghost in the Shell,0.039557,0.457537
1947,9408,Surf's Up,0.039009,0.397801
1961,360814,Dangal,0.035431,0.313259


## Ghi chú cho báo cáo
- Bước **Retrieval** cho ra danh sách ứng viên dựa trên TF-IDF (TF-IDF uni+bi-gram, `min_df=2`).
- Bước **ABSA** dự đoán nhãn aspect×sentiment trên **query** và lấy nhãn có xác suất cao nhất làm *target*.
- Sau đó, hệ thống chạy ABSA trên **review_profile** của top-N phim từ retrieval để **rerank**.


In [7]:
# =========================================================
# Demo thêm: Multi-field search + ABSA bonus rerank (4 use-cases)
# =========================================================
# Use-cases:
# A) Tên phim (có thể gõ sai nhẹ)  B) Thể loại  C) Ngữ cảnh dài  D) Query có cảm xúc/aspect

import json
import math
import re
from difflib import SequenceMatcher
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load thêm cột nếu có
extra_cols = [c for c in ["genres", "movie_profile"] if c in df.columns]
base_cols = ["tmdb_id", "title", "review_profile"]
demo_df = df[base_cols + extra_cols].copy()
if "genres" not in demo_df.columns:
    demo_df["genres"] = ""

# Build a robust movie_profile if missing
if "movie_profile" not in demo_df.columns:
    def _mk_profile(row):
        title = str(row.get("title", ""))
        genres = str(row.get("genres", ""))
        reviews = str(row.get("review_profile", ""))
        reviews = reviews[:600]
        return f"title: {title} | genres: {genres} | reviews: {reviews}"
    demo_df["movie_profile"] = demo_df.apply(_mk_profile, axis=1)

# --- Load ABSA movie profiles (exported from Notebook 04) ---
profiles_path = NOTEBOOK_DIR / "absa" / "absa_movie_profiles.json"
absa_profiles = {}
if profiles_path.exists():
    absa_profiles = json.loads(profiles_path.read_text(encoding="utf-8"))
    print("Loaded ABSA movie profiles:", len(absa_profiles))
else:
    print("ABSA movie profiles missing (run Notebook 04 to export). Will skip ABSA bonus.")

# --- Multi-field scoring ---

def _norm_tokens(s: str) -> list[str]:
    return [t for t in re.sub(r"[^a-zA-Z0-9\s]+", " ", str(s).lower()).split() if t]


def title_lexical_score(query: str, title: str) -> float:
    q = " ".join(_norm_tokens(query))
    t = " ".join(_norm_tokens(title))
    if not q or not t:
        return 0.0
    ratio = SequenceMatcher(None, q, t).ratio()
    qset = set(q.split())
    tset = set(t.split())
    overlap = (len(qset & tset) / max(1, len(qset))) if qset else 0.0
    return 0.6 * ratio + 0.4 * overlap


def genre_keyword_score(query: str, genres_str: str) -> float:
    q_tokens = set(_norm_tokens(query))
    if not q_tokens:
        return 0.0
    genres = {g.strip().lower() for g in str(genres_str).split(",") if g and g.strip()}
    if not genres:
        return 0.0
    genre_tokens = set()
    for g in genres:
        genre_tokens.update(_norm_tokens(g))
    hits = sum(1 for tok in q_tokens if tok in genre_tokens)
    return hits / max(1, len(q_tokens))


def choose_weights(query: str) -> tuple[float, float, float]:
    n = len(_norm_tokens(query))
    if n <= 3:
        return (1.0, 0.8, 0.3)
    if n <= 6:
        return (0.7, 0.8, 0.6)
    return (0.2, 0.8, 1.0)


# Semantic TF-IDF index on movie_profile
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, max_df=0.9)
X = vectorizer.fit_transform(demo_df["movie_profile"].astype(str).tolist())


def semantic_scores(query: str) -> np.ndarray:
    qv = vectorizer.transform([str(query)])
    sims = cosine_similarity(qv, X).ravel()
    mx = float(sims.max()) if sims.size else 0.0
    return sims / mx if mx > 0 else sims


# --- ABSA bonus heuristic mapping ---
ASPECT_KEYWORDS = {
    "overall": {"positive": ["hay", "xuất sắc", "good", "great", "amazing", "excellent"], "negative": ["tệ", "bad", "awful", "terrible"]},
    "script": {"positive": ["kịch bản hay", "plot twist", "story great", "script great", "mind-bending"], "negative": ["bad script", "plot hole", "boring story"]},
    "visuals": {"positive": ["kỹ xảo đẹp", "visuals", "cgi", "beautiful"], "negative": ["ugly", "bad visuals"]},
    "acting": {"positive": ["diễn xuất tốt", "acting", "performance"], "negative": ["bad acting"]},
    "pacing": {"positive": ["fast paced", "good pacing"], "negative": ["too slow", "slow", "drag", "boring"]},
    "music": {"positive": ["soundtrack", "music great", "nhạc hay"], "negative": ["bad music"]},
    "direction": {"positive": ["good direction", "director"], "negative": ["bad director"]},
}


def infer_query_intents(query: str) -> list[tuple[str, str]]:
    q = str(query).lower()
    intents = []
    for aspect, sd in ASPECT_KEYWORDS.items():
        for sent, kws in sd.items():
            if any(kw in q for kw in kws):
                intents.append((aspect, sent))
    return intents


def absa_bonus_for_movie(tmdb_id: str, intents: list[tuple[str, str]], scale: float = 20.0) -> float:
    if not absa_profiles or not intents:
        return 0.0
    rec = absa_profiles.get(str(tmdb_id))
    if not rec:
        return 0.0
    scores = rec.get("scores", {})

    bonus = 0.0
    for aspect, sent in intents:
        val = float(scores.get(aspect, {}).get(sent, 0.0))
        # positive: cộng điểm, negative: trừ điểm (nhẹ hơn để tránh quá tay)
        if sent == "negative":
            bonus -= val * (scale * 0.7)
        else:
            bonus += val * scale
    return float(bonus)


def run_case(query: str, top_k: int = 10):
    ss = semantic_scores(query)
    ts = demo_df["title"].apply(lambda t: title_lexical_score(query, t)).to_numpy(dtype=float)
    gs = demo_df["genres"].apply(lambda g: genre_keyword_score(query, g)).to_numpy(dtype=float)

    w_title, w_genre, w_sem = choose_weights(query)
    fused = (w_title * ts) + (w_genre * gs) + (w_sem * ss)

    intents = infer_query_intents(query)
    bonus = np.array([absa_bonus_for_movie(mid, intents) for mid in demo_df["tmdb_id"].astype(str).tolist()], dtype=float)
    final = fused + bonus

    out = demo_df[["tmdb_id", "title", "genres"]].copy()
    out["score_title"] = ts
    out["score_genre"] = gs
    out["score_semantic"] = ss
    out["score_fusion"] = fused
    out["absa_bonus"] = bonus
    out["score_final"] = final

    # show before/after
    top_before = out.sort_values("score_fusion", ascending=False).head(top_k)
    top_after = out.sort_values("score_final", ascending=False).head(top_k)

    print("\n---")
    print("Query:", query)
    print("Weights:", f"title={w_title}, genre={w_genre}, semantic={w_sem}")
    print("Intents:", intents)
    print("\nTop BEFORE (fusion):")
    display(top_before)
    print("\nTop AFTER (fusion + ABSA bonus):")
    display(top_after)


# 4 demo cases
cases = {
    "A_title_typo": "Incpetion",  # typo
    "B_genre": "action horror",
    "C_context": "mind-bending dream and time loop",
    "D_aspect_sentiment": "great visuals but pacing was too slow",
}

for name, q in cases.items():
    print(f"\n=== Case {name} ===")
    run_case(q, top_k=8)


Loaded ABSA movie profiles: 3230

=== Case A_title_typo ===

---
Query: Incpetion
Weights: title=1.0, genre=0.8, semantic=0.3
Intents: []

Top BEFORE (fusion):


,tmdb_id,title,genres,score_title,score_genre,score_semantic,score_fusion,absa_bonus,score_final
103,27205,Inception,"Action, Adventure, Science Fiction",0.533333,0.0,0.0,0.533333,0.0,0.533333
2805,429415,Extinction,"Action, Drama, Science Fiction, Thriller",0.442105,0.0,0.0,0.442105,0.0,0.442105
1429,715931,Emancipation,"Drama, History, War",0.400000,0.0,0.0,0.400000,0.0,0.400000
2026,211672,Minions,"Adventure, Animation, Comedy, Family",0.375000,0.0,0.0,0.375000,0.0,0.375000
1535,72976,Lincoln,"Drama, History",0.375000,0.0,0.0,0.375000,0.0,0.375000
1793,993710,Back in Action,"Action, Comedy",0.365217,0.0,0.0,0.365217,0.0,0.365217
2814,831946,Interceptor,"Action, Adventure, Thriller",0.360000,0.0,0.0,0.360000,0.0,0.360000
2688,11075,Audition,"Drama, Horror",0.352941,0.0,0.0,0.352941,0.0,0.352941



Top AFTER (fusion + ABSA bonus):


,tmdb_id,title,genres,score_title,score_genre,score_semantic,score_fusion,absa_bonus,score_final
103,27205,Inception,"Action, Adventure, Science Fiction",0.533333,0.0,0.0,0.533333,0.0,0.533333
2805,429415,Extinction,"Action, Drama, Science Fiction, Thriller",0.442105,0.0,0.0,0.442105,0.0,0.442105
1429,715931,Emancipation,"Drama, History, War",0.400000,0.0,0.0,0.400000,0.0,0.400000
2026,211672,Minions,"Adventure, Animation, Comedy, Family",0.375000,0.0,0.0,0.375000,0.0,0.375000
1535,72976,Lincoln,"Drama, History",0.375000,0.0,0.0,0.375000,0.0,0.375000
1793,993710,Back in Action,"Action, Comedy",0.365217,0.0,0.0,0.365217,0.0,0.365217
2814,831946,Interceptor,"Action, Adventure, Thriller",0.360000,0.0,0.0,0.360000,0.0,0.360000
2688,11075,Audition,"Drama, Horror",0.352941,0.0,0.0,0.352941,0.0,0.352941



=== Case B_genre ===

---
Query: action horror
Weights: title=1.0, genre=0.8, semantic=0.3
Intents: []

Top BEFORE (fusion):


...

,tmdb_id,title,genres,score_title,score_genre,score_semantic,score_fusion,absa_bonus,score_final
1336,4564,Sex and the City,"Comedy, Drama, Romance",0.291667,0.0,0.000000,0.204167,19.816,20.020167
2053,320956,The Cuckold,Drama,0.111628,0.0,0.000000,0.078140,19.936,20.014140
1495,1289936,Downton Abbey: The Grand Finale,"Drama, Romance",0.232258,0.0,0.011458,0.169455,19.840,20.009455
1650,11860,Sabrina,"Comedy, Drama, Romance",0.092308,0.0,0.000000,0.064615,19.926,19.990615
2004,783,Gandhi,"Drama, History",0.157895,0.0,0.000000,0.110526,19.876,19.986526
1468,454527,Benedetta,"Drama, History, Romance",0.175610,0.0,0.000000,0.122927,19.854,19.976927
3118,515248,Someone Great,"Comedy, Romance",0.213333,0.0,0.047642,0.177919,19.792,19.969919
3188,16633,Somewhere in Time,"Drama, Fantasy, Romance",0.262585,0.0,0.028571,0.200952,19.766,19.966952



=== Case D_aspect_sentiment ===

---
Query: great visuals but pacing was too slow
Weights: title=0.2, genre=0.8, semantic=1.0
Intents: [('overall', 'positive'), ('visuals', 'positive'), ('pacing', 'negative')]

Top BEFORE (fusion):


,tmdb_id,title,genres,score_title,score_genre,score_semantic,score_fusion,absa_bonus,score_final
385,62,2001: A Space Odyssey,"Adventure, Mystery, Science Fiction",0.189474,0.0,1.000000,1.037895,14.6714,15.709295
2040,638,Lost Highway,"Drama, Mystery, Thriller",0.146939,0.0,0.982883,1.012271,38.1216,39.133871
1947,9408,Surf's Up,"Animation, Comedy, Family",0.156522,0.0,0.727459,0.758763,20.3186,21.077363
188,315837,Ghost in the Shell,"Action, Drama, Science Fiction",0.218182,0.0,0.704575,0.748211,15.0750,15.823211
281,762509,Mufasa: The Lion King,"Adventure, Animation, Family",0.147368,0.0,0.710981,0.740454,38.8740,39.614454
94,693134,Dune: Part Two,"Adventure, Science Fiction",0.168000,0.0,0.631018,0.664618,37.0000,37.664618
3174,664469,Amsterdam,"Comedy, Drama, Mystery",0.052174,0.0,0.615759,0.626194,28.8188,29.444994
673,2048,"I, Robot","Action, Science Fiction",0.054545,0.0,0.608289,0.619198,24.4584,25.077598



Top AFTER (fusion + ABSA bonus):


,tmdb_id,title,genres,score_title,score_genre,score_semantic,score_fusion,absa_bonus,score_final
1126,1608982,From the Ashes: The Pit,"Drama, Thriller",0.162712,0.0,0.405536,0.438078,39.5070,39.945078
2344,975335,Magellan,"Adventure, Drama, History",0.106667,0.0,0.166989,0.188322,39.4280,39.616322
281,762509,Mufasa: The Lion King,"Adventure, Animation, Family",0.147368,0.0,0.710981,0.740454,38.8740,39.614454
2066,1904,Memoirs of a Geisha,"Drama, History, Romance",0.107143,0.0,0.217586,0.239014,39.3294,39.568414
2944,491480,The Boy Who Harnessed the Wind,"Drama, History",0.125373,0.0,0.175056,0.200131,39.3386,39.538731
1463,610150,Dragon Ball Super: Super Hero,"Action, Animation, Science Fiction",0.166154,0.0,0.175050,0.208281,39.3230,39.531281
1884,25237,Come and See,"Drama, War",0.122449,0.0,0.031750,0.056240,39.4740,39.530240
2063,888,The Flintstones,"Comedy, Family, Fantasy",0.207692,0.0,0.041567,0.083106,39.4174,39.500506


## Future Work (Phase 2) — Cá nhân hoá theo User (tổng hợp từ `nguyen.md`)

Demo hiện tại tập trung vào **discovery/recommendation theo nội dung** (movie profile + review text) và **ABSA explainability**. Để tiến tới *cá nhân hoá thật sự* theo từng người dùng, có thể mở rộng theo blueprint sau:

- **Bổ sung dữ liệu user nội bộ**
  - Thêm bảng `user_reviews`: user viết review trực tiếp trên web/app.
  - Thêm bảng `absa_user_profiles`: lưu vector “gu” theo khía cạnh (tỉ trọng thích/chê `acting/script/visuals/...`).

- **Offline/batch pipeline để sinh profile**
  - **Movie ABSA profile**: aggregate ABSA trên toàn bộ review của phim (TMDB + user review).
  - **User ABSA profile**: aggregate ABSA trên toàn bộ review user từng viết (có thể weighted theo thời gian).

- **Hybrid recommender (2 luồng + fusion)**
  - **Luồng 1 — Content-based (ABSA matching)**: cosine similarity giữa `user_profile_vector` và `movie_profile_vector`.
  - **Luồng 2 — Collaborative (user-based)**: tìm “tri kỷ” có gu giống nhau rồi gợi ý phim họ thích mà user chưa xem.
  - **Fusion**: `FinalScore = α*Score_content + β*Score_collab`.

- **Giải cold-start (onboarding)**
  - UI hỏi user ưu tiên aspect nào (script/acting/visuals/...) để khởi tạo profile ban đầu.

- **Tích hợp API/UI**
  - Endpoint gợi ý cá nhân hoá: `GET /users/{user_id}/recommendations`.
  - UI “For You”: hiển thị badge giải thích: *“Match 96% với gu Script & Visuals của bạn”*.

Ghi chú: Phase 2 cần thay đổi schema DB + data collection cho user, nên nên để ở phần mở rộng nếu phạm vi đồ án hiện tại ưu tiên demo end-to-end theo barem.
